In [ ]:
#original code from ChatGPT

In [2]:
import rasterio
import fiona
from rasterio.features import geometry_mask

def generate_binary_masks(image_path, shapefile_path, output_path_template):
    # Open the satellite image file
    with rasterio.open(image_path) as src:
        # Read the image as a numpy array
        image = src.read()

    # Open the shapefile
    with fiona.open(shapefile_path, "r") as shapefile:
        for i, feature in enumerate(shapefile):
            # Get the geometry of the feature
            geometry = feature["geometry"]

            # Generate a mask from the geometry that has the same shape as the image
            mask = geometry_mask([geometry], out_shape=image.shape[1:], transform=src.transform, invert=True)

            # Save the mask to a new GeoTIFF file
            output_path = output_path_template.format(i)
            with rasterio.open(output_path, "w", driver="GTiff", height=image.shape[1], width=image.shape[2], count=1, dtype="uint8", crs=src.crs, transform=src.transform) as dst:
                dst.write(mask.astype("uint8"), 1)

In [ ]:
# example
# generate_binary_masks("path/to/image.tif", "path/to/shapefile.shp", "path/to/output_mask_{}.tif")


In [ ]:
# code adjust

In [1]:
import rasterio
import fiona
from rasterio.features import geometry_mask

In [3]:
generate_binary_masks(r"D:\deep_learning\Sentinel_forestfire\MY_TEST_SENTINEL_IMAGERY_DATA\Google_Earth_Engine\temp\T52SDE_20220114T021041_2022001.tif", r"D:\deep_learning\Sentinel_forestfire\MY_TEST_SENTINEL_IMAGERY_DATA\Google_Earth_Engine\polygons_for_firemasks\2022001_2022030.shp", r"D:\deep_learning\Sentinel_forestfire\MY_TEST_SENTINEL_IMAGERY_DATA\Google_Earth_Engine\output_folder\output_mask_{}.tif")

In [2]:
from glob import glob
import os

In [3]:
os.chdir(r'D:\deep_learning\Sentinel_forestfire\MY_TEST_SENTINEL_IMAGERY_DATA\Google_Earth_Engine')

In [4]:
shp = fiona.open('../Fire_boundary_polygon_files/2021/2021001_2021028.shp')

In [8]:
for i, feature in enumerate(shp):
    print(feature)
    if i == 2:
        break

{'type': 'Feature', 'id': '0', 'properties': OrderedDict([('Fire_masks', 1), ('Fire_IDs', '2021001'), ('I_date', '2021-01-04'), ('Sub_ID', '1'), ('ID', 1), ('Condition', None)]), 'geometry': {'type': 'MultiPolygon', 'coordinates': [[[(388429.50847371947, 3923029.627867302), (388429.50847371947, 3923011.1069969237), (388459.93561790884, 3923011.1069969237), (388459.2741582524, 3922999.531452939), (388469.5267829262, 3922999.531452939), (388470.51897240896, 3922990.2710177545), (388480.11013742536, 3922989.9402879328), (388480.11013742536, 3922979.6876632534), (388469.5267829262, 3922980.3491229154), (388470.1882425798, 3922970.4272280727), (388469.692147837, 3922960.2716174796), (388459.6048880797, 3922960.00703362), (388459.6048880797, 3922930.4089188613), (388449.6829932388, 3922930.7396486923), (388450.3444528952, 3922909.9036695193), (388439.7610983951, 3922910.5651291795), (388439.7610983951, 3922880.799444644), (388399.7427891884, 3922881.4609042965), (388400.40424884483, 3922900.

In [9]:
import os
from glob import glob
import rasterio
import fiona
from rasterio.features import geometry_mask

In [41]:
#set default directory
os.chdir(r'D:\deep_learning\Sentinel_forestfire\MY_TEST_SENTINEL_IMAGERY_DATA\Google_Earth_Engine')

image_path = '2021/temp/image'
shapefile_path = '../Fire_boundary_polygon_files/2021'

In [42]:
# list satellite images using glob (list)
images = glob(f'{image_path}/*.tif')
# list shapefiles using glob (list)
shps = glob(f'{shapefile_path}/*.shp')


In [44]:
output_path = '2021/temp/mask'

In [45]:
len(images)

36

In [46]:
images[5][16:-4]

'T52SCD_20211228T022111_2021027'

In [47]:
images[5][23:31]

'20211228'

In [48]:
shps

['../Fire_boundary_polygon_files/2021\\2021001_2021028.shp']

In [49]:
shps[0][-11:-4]

'2021028'

In [51]:

#select first image data through loop
for inum in images:
    with rasterio.open(inum) as src:
        #read the image as numpy array
        image = src.read()
        
        Image_FID = int(inum[39:46])
        Image_Date = int(inum[23:31])     
        
        
        for snum in shps:
             
            #shp_range_text = snum[-19:-4]
            shp_s_range = int(snum[-19:-12])
            shp_e_range = int(snum[-11:-4])
            shp_id_list = list(range(shp_s_range, shp_e_range+1))
            
            if Image_FID not in shp_id_list:
                print(Image_FID)
                continue
            else:
                with fiona.open(snum) as shapefile:
                    for feature in shapefile:
                        if feature['properties']['Fire_IDs']==str(Image_FID) and feature['properties']['I_date'].replace('-','')==str(Image_Date):
                            target_geometry = feature["geometry"]
                            
                            # Generate a mask from the geometry that has the same shape as the image
                            mask = geometry_mask([target_geometry], out_shape=image.shape[1:], transform=src.transform, invert=True)
                            
                            # Save the mask to a new GeoTIFF file
                            
                            output_file = f"{output_path}/{inum[16:-4]}_mask.tif"
                            with rasterio.open(output_file, "w", driver="GTiff", height=image.shape[1], width=image.shape[2], count=1, dtype="uint8", crs=src.crs, transform=src.transform) as dst:
                                dst.write(mask.astype("uint8"), 1)


In [66]:
for b in images:
    print(f'{b[5:-4]}_mask.tif')

T52SBD_20220402T021559_2022046_mask.tif
T52SBE_20220522T021609_2022077_mask.tif
T52SBF_20220303T021609_2022028_mask.tif
T52SBF_20220308T021611_2022038_mask.tif
T52SBF_20220417T021611_2022058_mask.tif
T52SCD_20220203T020911_2022003_mask.tif
T52SCD_20220226T021651_2022022_mask.tif
T52SCD_20220226T021651_2022023_mask.tif
T52SCD_20220412T021559_2022055_mask.tif
T52SCE_20220226T021651_2022021_mask.tif
T52SCE_20220305T020701_2022033_mask.tif
T52SCE_20220315T020701_2022047_mask.tif
T52SCE_20220404T020701_2022049_mask.tif
T52SCE_20220502T021559_2022073_mask.tif
T52SCF_20220226T021651_2022012_mask.tif
T52SCF_20220417T021611_2022054_mask.tif
T52SCF_20220424T020701_2022072_mask.tif
T52SCF_20221208T022059_2022084_mask.tif
T52SCGF_20220427T021611_2022069_mask.tif
T52SCG_20220226T021651_2022018_mask.tif
T52SCG_20220308T021611_2022034_mask.tif
T52SCG_20220308T021611_2022040_mask.tif
T52SCG_20220328T021611_2022043_mask.tif
T52SCG_20220407T021601_2022050_mask.tif
T52SCG_20220412T021559_2022056_mask.tif

In [ ]:
## big file adjustment

In [1]:
import os
from glob import glob
import rasterio
import fiona
from rasterio.features import geometry_mask

In [2]:
#set default directory
os.chdir(r'D:\deep_learning\Sentinel_forestfire\MY_TEST_SENTINEL_IMAGERY_DATA\Google_Earth_Engine')

image_path = 'temp/big_file'
shapefile_path = 'polygons_for_firemasks'

In [3]:
# list satellite images using glob (list)
images = glob(f'{image_path}/*.tif')
# list shapefiles using glob (list)
shps = glob(f'{shapefile_path}/*.shp')
output_path = 'output_folder'

In [4]:

#select first image data through loop
for inum in images:
    with rasterio.open(inum) as src:
        #read the image as numpy array
        image = src.read()
        
        Image_FID = int(inum[37:44])
        Image_Date = int(inum[21:29])     
        
        
        for snum in shps:
             
            #shp_range_text = snum[-19:-4]
            shp_s_range = int(snum[-19:-12])
            shp_e_range = int(snum[-11:-4])
            shp_id_list = list(range(shp_s_range, shp_e_range+1))
            
            if Image_FID not in shp_id_list:
                continue
            else:
                with fiona.open(snum) as shapefile:
                    for feature in shapefile:
                        if feature['properties']['Fire_IDs']==str(Image_FID) and feature['properties']['I_date'].replace('-','')==str(Image_Date):
                            target_geometry = feature["geometry"]
                            
                            # Generate a mask from the geometry that has the same shape as the image
                            mask = geometry_mask([target_geometry], out_shape=image.shape[1:], transform=src.transform, invert=True)
                            
                            # Save the mask to a new GeoTIFF file
                            
                            output_file = f"{output_path}/{inum[5:-4]}_mask.tif"
                            with rasterio.open(output_file, "w", driver="GTiff", height=image.shape[1], width=image.shape[2], count=1, dtype="uint8", crs=src.crs, transform=src.transform) as dst:
                                dst.write(mask.astype("uint8"), 1)
